## Desafio D — Sistema de Registro de Preços

### Problema

Quais características diferenciam contratações realizadas com e sem Sistema de Registro de Preços?

### Possíveis perguntas

- O SRP é mais frequente em determinados tipos de contratação?
- Existem diferenças nos valores das contratações?
- Determinados órgãos utilizam SRP proporcionalmente mais do que outros?

### Variável de interesse

Quando disponível:

```text
srp
```

### Possíveis análises

- proporções;
- tabelas cruzadas;
- comparação de valores;
- teste qui-quadrado.


documentação API: https://dadosabertos.compras.gov.br/swagger-ui/index.html


## Instalação e importação das bibliotecas utilizadas

In [1]:
#!pip install requests pandas matplotlib -q
#!pip install pyarrow

In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time

## Pegando a base do ENDPOINT para a consulta da API

In [3]:
BASE_URL = "https://dadosabertos.compras.gov.br"

ENDPOINT_CONTRATACOES = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"

url = BASE_URL + ENDPOINT_CONTRATACOES

print(url)

https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133


## Criando função para extrair os registros e dados da API

In [ ]:
def extrair_registros(json_resposta):

    if isinstance(json_resposta, list): # Se a reposta ja for uma lista, ele apenas retorna ela
        return json_resposta

    if not isinstance(json_resposta, dict):  # Se a reposta nao for uma lista e nem um dicionario, ele apenas retorna uma lista vazia para reagir a respostas inesperadas
        return []

    for chave in ["resultado", "resultados", "data", "content"]: # Checa se no dicionario as  inormações vieram por esse nome
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave] # Se veio essas chaves e se o valor associado a ele tambem veio, retorna os dados

    return [] # Se ele não achar nada, retorna a lista vazia



# Coleta de Dados

## Escolha dos dados
A seguir serão coletados os dados referente ao ano de 2024 e 2025, nas modalidades de pregão-eletronico e dispensa, com o objetivo de analisar o SRP e fazer uma analise de dados 

## Coletando dados de 2024

In [5]:
modalidades = [5, 6] # Pega principais modalidades
todos_registros = []

In [6]:
for modalidade in modalidades:
    pagina = 1
    total_modalidade = 0

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": 500,
            "dataPublicacaoPncpInicial": "2024-01-01",
            "dataPublicacaoPncpFinal": "2024-12-31",
            "codigoModalidade": modalidade
        }

        resposta = requests.get(url, params=params, timeout=60)

        if resposta.status_code == 429:
            espera = 5
            print(f"Modalidade {modalidade}, página {pagina}: 429, aguardando {espera}s e tentando de novo...")
            time.sleep(espera)
            continue  # tenta a mesma página de novo, sem avançar

        if resposta.status_code != 200:
            print(f"Modalidade {modalidade}, página {pagina}: erro {resposta.status_code}")
            break

        dados = resposta.json()
        registros = extrair_registros(dados)

        if not registros:
            break

        todos_registros.extend(registros)
        total_modalidade += len(registros)

        if len(registros) < 500:
            break

        pagina += 1
        time.sleep(0.2)  # pausa maior entre páginas

    print(f"Modalidade {modalidade}: {total_modalidade} registros coletados.")
    time.sleep(3)  # pausa entre modalidades, para o servidor "esfriar"

Modalidade 5: 104142 registros coletados.
Modalidade 6: 133930 registros coletados.


In [ ]:
df_24 = pd.json_normalize(todos_registros)
df_24.head()

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,Edital,Aberto-Fechado,614963.83,343793.5,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False
1,41300105000262023,02030715000112-1-000226/2023,2023,226,02030715000112,NaN,89804,AGENCIA NACIONAL DE TELECOMUNICACOES,NaN,F,...,Edital,Aberto-Fechado,404.74,NaN,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T08:00:00,2024-01-17T10:00:00,False
2,15812605000492023,10729992000146-1-000127/2023,2023,127,10729992000146,NaN,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",NaN,F,...,Edital,Aberto-Fechado,105000.00,105000.0,2024-01-02T07:00:05,2024-01-15T07:07:29,2024-01-02T07:00:05,2024-01-02T08:00:00,2024-01-16T10:00:00,False
3,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,Edital,Aberto,286011.82,270000.0,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False
4,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,Edital,Aberto,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False


In [ ]:

df_24["ano"] = 2024 # Adiciona a coluna "ano" a fim de saber o ano da contribuição

df_24.to_parquet("raw_completo_2024.parquet", index=False) # Salva parquet de 2024


if 'srp' in df_24.columns: # Verifica se o SRP veio na tabela
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_24['srp'].value_counts(dropna=False)) # Vê quantos valores diferentes vieram para verificar valores nulos

    # Cria uma tabela auxiliar com os dados de 2024
    df_24_com_srp = df_24[df_24['srp'] == True] 
    df_24_sem_srp = df_24[df_24['srp'] == False]

    # Mostra quantos valores vieram em cada um
    print(f"\n Total COM SRP: {len(df_24_com_srp)}")
    print(f" Total SEM SRP: {len(df_24_sem_srp)}")



Valores brutos encontrados na coluna SRP:
srp
False    194102
True      43970
Name: count, dtype: int64

 Total COM SRP: 43970
 Total SEM SRP: 194102


In [ ]:
# Mostra os dataframes criados
print("\nDataFrame com SRP:")
display(df_24_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_24_sem_srp.head())


DataFrame com SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,Aberto-Fechado,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2024
7,92804805900012024,88830609000139-1-000001/2024,2024,1,88830609000139,NaN,41243,MUNICIPIO DE CAXIAS DO SUL,NaN,M,...,Aberto,68469.70,27300.60,2024-01-02T07:00:16,2024-01-02T07:00:16,2024-01-02T07:00:16,2024-01-02T08:00:00,2024-01-16T08:30:00,False,2024
9,15590105000962023,15126437000143-1-003231/2023,2023,3231,15126437000143,NaN,95159,EMPRESA BRASILEIRA DE SERVIÇOS HOSPITALARES,NaN,F,...,Aberto-Fechado,137987.47,103025.62,2024-01-02T07:00:20,2024-01-02T07:00:20,2024-01-02T07:00:20,2024-01-02T08:00:00,2024-01-12T09:00:00,False,2024
11,15812605000352023,10729992000146-1-000128/2023,2023,128,10729992000146,NaN,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",NaN,F,...,Aberto,13225.77,1600.00,2024-01-02T07:00:23,2024-01-02T07:00:23,2024-01-02T07:00:23,2024-01-02T08:00:00,2024-01-22T10:00:00,False,2024
12,16039905000432023,00394452000103-1-014522/2023,2023,14522,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,...,Aberto-Fechado,6389632.82,1145924.50,2024-01-02T07:00:24,2024-01-05T07:04:20,2024-01-02T07:00:24,2024-01-05T08:00:00,2024-01-17T09:00:00,False,2024



DataFrame sem SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano
1,41300105000262023,02030715000112-1-000226/2023,2023,226,02030715000112,NaN,89804,AGENCIA NACIONAL DE TELECOMUNICACOES,NaN,F,...,Aberto-Fechado,404.74,NaN,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T08:00:00,2024-01-17T10:00:00,False,2024
2,15812605000492023,10729992000146-1-000127/2023,2023,127,10729992000146,NaN,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",NaN,F,...,Aberto-Fechado,105000.00,105000.0,2024-01-02T07:00:05,2024-01-15T07:07:29,2024-01-02T07:00:05,2024-01-02T08:00:00,2024-01-16T10:00:00,False,2024
3,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,Aberto,286011.82,270000.0,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2024
4,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,Aberto,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2024
5,92668605000222023,04801221000110-1-000444/2023,2023,444,04801221000110,NaN,53800,TRIBUNAL DE CONTAS DO ESTADO DE RONDONIA,NaN,E,...,Aberto,99975.00,95466.0,2024-01-02T07:00:10,2024-01-02T07:00:10,2024-01-02T07:00:10,2024-01-02T08:00:00,2024-01-16T10:00:00,False,2024


In [ ]:
# Dataframe utiliando GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_24 = df_24.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

df_qnt_modalidade_orgao_24.index = ['Sem SRP', 'Com SRP'] # Renomeia os índices para facilitar a leitura no relatório

display(df_qnt_modalidade_orgao_24)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,2463,194102
Com SRP,3,1423,43970


# Pegando dados 2025

In [11]:
modalidades = [5, 6] # Pega principais modalidades
todos_registros = []

In [12]:
for modalidade in modalidades:
    pagina = 1
    total_modalidade = 0

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": 500,
            "dataPublicacaoPncpInicial": "2025-01-01",
            "dataPublicacaoPncpFinal": "2025-12-31",
            "codigoModalidade": modalidade,
        }

        resposta = requests.get(url, params=params, timeout=60)

        if resposta.status_code == 429:
            espera = 5
            print(f"Modalidade {modalidade}, página {pagina}: 429, aguardando {espera}s e tentando de novo...")
            time.sleep(espera)
            continue  # tenta a mesma página de novo, sem avançar

        if resposta.status_code != 200:
            print(f"Modalidade {modalidade}, página {pagina}: erro {resposta.status_code}")
            break

        dados = resposta.json()
        registros = extrair_registros(dados)

        if not registros:
            break

        todos_registros.extend(registros)
        total_modalidade += len(registros)

        if len(registros) < 500:
            break

        pagina += 1
        time.sleep(0.2)  # pausa maior entre páginas

    print(f"Modalidade {modalidade}: {total_modalidade} registros coletados.")
    time.sleep(3)  # pausa entre modalidades, para o servidor "esfriar"

Modalidade 5: 114824 registros coletados.
Modalidade 6: 131369 registros coletados.


In [ ]:
df_25 = pd.json_normalize(todos_registros)
df_25.head()

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,92647305900242024,06272868000127-1-000056/2024,2024,56,06272868000127,NaN,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,NaN,F,...,Edital,Aberto-Fechado,400000.00,NaN,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T08:00:00,2025-01-16T09:00:00,False
1,92647305900252024,06272868000127-1-000057/2024,2024,57,06272868000127,NaN,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,NaN,F,...,Edital,Aberto-Fechado,250782.10,151826.000,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T08:00:00,2025-01-14T09:00:00,False
2,38910305900162024,21947619000188-1-000068/2024,2024,68,21947619000188,NaN,21599,CONSELHO REGIONAL DE FISIOTERAPIA E TERAPIA OC...,NaN,F,...,Edital,Aberto-Fechado,868719.38,NaN,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:00:00,2025-01-17T09:00:00,False
3,94300105913862024,07954480000179-1-023839/2024,2024,23839,07954480000179,NaN,57870,ESTADO DO CEARA,NaN,E,...,Edital,Aberto-Fechado,11147211.58,6619438.535,2025-01-02T09:03:47,2025-05-09T08:13:51,2025-01-02T09:03:47,2025-04-25T08:00:00,2025-05-15T09:00:00,False
4,38918505900112024,00119784000171-1-000036/2024,2024,36,00119784000171,NaN,43167,CONSELHO FEDERAL DE MEDICINA VETERINARIA,NaN,F,...,Edital,Aberto,4612985.04,3718205.520,2025-01-02T09:03:51,2025-09-22T07:36:06,2025-01-02T09:03:51,2025-09-22T08:00:00,2025-10-07T10:00:00,False


In [ ]:
df_25["ano"] = 2025 # Adiciona a coluna "ano" a fim de saber o ano da contribuição

df_25.to_parquet("raw_completo_2025.parquet", index=False) # Salva parquet de 2025


if 'srp' in df_25.columns: # Verifica se o SRP veio na tabela
    
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_25['srp'].value_counts(dropna=False))  # Vê quantos valores diferentes vieram para verificar valores nulos


    # Cria uma tabela auxiliar com os dados de 2025
    df_25_com_srp = df_25[df_25['srp'] == True]
    df_25_sem_srp = df_25[df_25['srp'] == False]

    # Mostra quantos valores vieram em cada um
    print(f"\n Total COM SRP: {len(df_25_com_srp)}")
    print(f" Total SEM SRP: {len(df_25_sem_srp)}")


Valores brutos encontrados na coluna SRP:
srp
False    193987
True      52206
Name: count, dtype: int64

 Total COM SRP: 52206
 Total SEM SRP: 193987


In [15]:
print("\nDataFrame com SRP:")
display(df_25_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_25_sem_srp.head())


DataFrame com SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano
5,94300105914952024,07954480000179-1-023841/2024,2024,23841,07954480000179,NaN,57870,ESTADO DO CEARA,NaN,E,...,Aberto-Fechado,138742.18,89250.00,2025-01-02T09:03:57,2025-01-02T09:03:57,2025-01-02T09:03:57,2025-01-03T08:00:00,2025-01-15T14:30:00,False,2025
7,94300105913952024,07954480000179-1-023842/2024,2024,23842,07954480000179,NaN,57870,ESTADO DO CEARA,NaN,E,...,Aberto-Fechado,759632.82,551284.00,2025-01-02T09:04:04,2025-01-02T09:04:04,2025-01-02T09:04:04,2025-01-03T08:00:00,2025-01-15T14:30:00,False,2025
8,94300105910162024,07954480000179-1-023843/2024,2024,23843,07954480000179,NaN,57870,ESTADO DO CEARA,NaN,E,...,Aberto-Fechado,55242.57,37068.54,2025-01-02T09:04:07,2025-01-02T09:04:07,2025-01-02T09:04:07,2025-01-03T08:00:00,2025-01-15T09:00:00,False,2025
10,98748705900602024,75972760000160-1-000185/2024,2024,185,75972760000160,NaN,84922,MUNICIPIO DE CAPANEMA,NaN,M,...,Aberto,302156.25,206660.50,2025-01-02T09:04:14,2025-01-17T07:48:19,2025-01-02T09:04:14,2025-01-02T08:00:00,2025-01-21T08:30:00,False,2025
14,42512805900152024,02973240000106-1-000038/2024,2024,38,02973240000106,NaN,51122,ESTADO DO MARANHAO - SECRETARIA DE ESTADO DA S...,NaN,E,...,Aberto-Fechado,4425190.08,2782461.00,2025-01-02T09:04:26,2025-01-02T09:04:26,2025-01-02T09:04:26,2025-01-02T08:00:00,2025-01-15T09:00:00,False,2025



DataFrame sem SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano
0,92647305900242024,06272868000127-1-000056/2024,2024,56,06272868000127,NaN,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,NaN,F,...,Aberto-Fechado,400000.00,NaN,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T08:00:00,2025-01-16T09:00:00,False,2025
1,92647305900252024,06272868000127-1-000057/2024,2024,57,06272868000127,NaN,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,NaN,F,...,Aberto-Fechado,250782.10,151826.000,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T08:00:00,2025-01-14T09:00:00,False,2025
2,38910305900162024,21947619000188-1-000068/2024,2024,68,21947619000188,NaN,21599,CONSELHO REGIONAL DE FISIOTERAPIA E TERAPIA OC...,NaN,F,...,Aberto-Fechado,868719.38,NaN,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:00:00,2025-01-17T09:00:00,False,2025
3,94300105913862024,07954480000179-1-023839/2024,2024,23839,07954480000179,NaN,57870,ESTADO DO CEARA,NaN,E,...,Aberto-Fechado,11147211.58,6619438.535,2025-01-02T09:03:47,2025-05-09T08:13:51,2025-01-02T09:03:47,2025-04-25T08:00:00,2025-05-15T09:00:00,False,2025
4,38918505900112024,00119784000171-1-000036/2024,2024,36,00119784000171,NaN,43167,CONSELHO FEDERAL DE MEDICINA VETERINARIA,NaN,F,...,Aberto,4612985.04,3718205.520,2025-01-02T09:03:51,2025-09-22T07:36:06,2025-01-02T09:03:51,2025-09-22T08:00:00,2025-10-07T10:00:00,False,2025


In [ ]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_25 = df_25.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

df_qnt_modalidade_orgao_25.index = ['Sem SRP', 'Com SRP'] # Renomeia os índices para facilitar a leitura no relatório

display(df_qnt_modalidade_orgao_25)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,2871,193987
Com SRP,3,1646,52206


# Comparação 2024 X 2025

## Comparação das tabelas auxiliares 

In [53]:
display(df_qnt_modalidade_orgao_24)
display(df_qnt_modalidade_orgao_25)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,2463,194102
Com SRP,3,1423,43970


,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,2871,193987
Com SRP,3,1646,52206


### Gera um dataframe final com a junção dos dados de 2024 e 2025

In [54]:
df_total = pd.concat([df_24, df_25], ignore_index=True)
df_total

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,Aberto-Fechado,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2024
1,41300105000262023,02030715000112-1-000226/2023,2023,226,02030715000112,NaN,89804,AGENCIA NACIONAL DE TELECOMUNICACOES,NaN,F,...,Aberto-Fechado,404.74,NaN,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T08:00:00,2024-01-17T10:00:00,False,2024
2,15812605000492023,10729992000146-1-000127/2023,2023,127,10729992000146,NaN,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",NaN,F,...,Aberto-Fechado,105000.00,105000.00,2024-01-02T07:00:05,2024-01-15T07:07:29,2024-01-02T07:00:05,2024-01-02T08:00:00,2024-01-16T10:00:00,False,2024
3,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,Aberto,286011.82,270000.00,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2024
4,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,Aberto,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484260,16013706000462025,00394452000103-1-024933/2025,2025,24933,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,...,Não se aplica,2040.48,2040.48,2025-12-30T20:15:02,2025-12-30T20:15:02,2025-12-30T20:15:02,NaN,NaN,False,2025
484261,13500206000312025,00348003002245-1-000544/2025,2025,544,00348003002245,NaN,97892,EMPRESA BRASILEIRA DE PESQUISA AGROPECUARIA,NaN,N,...,Não se aplica,157970.43,157970.43,2025-12-30T20:19:40,2025-12-30T20:19:40,2025-12-30T20:19:40,NaN,NaN,False,2025
484262,15814306001252025,10791831000182-1-000053/2025,2025,53,10791831000182,NaN,60822,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",NaN,F,...,Não se aplica,1522362.00,1522362.00,2025-12-30T21:22:29,2025-12-30T21:22:29,2025-12-30T21:22:29,NaN,NaN,False,2025
484263,15814306001262025,10791831000182-1-000054/2025,2025,54,10791831000182,NaN,60822,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",NaN,F,...,Não se aplica,252900.00,252900.00,2025-12-30T21:35:41,2025-12-30T21:35:41,2025-12-30T21:35:41,NaN,NaN,False,2025


# Checamos duplicatas

In [55]:
# >>> AQUI ENTRA A CHECAGEM DE DUPLICATAS <
chave_dedup = [c for c in ['numeroCompra', 'orgaoEntidadeRazaoSocial', 'ano', 'modalidadeNome']
               if c in df_total.columns]
n_duplicados = df_total.duplicated(subset=chave_dedup).sum()
print(f"Duplicatas encontradas: {n_duplicados}")

if n_duplicados > 0:
    df_total = df_total.drop_duplicates(subset=chave_dedup)
    print(f"Total após remoção: {len(df_total)} registros")

Duplicatas encontradas: 215290
Total após remoção: 268975 registros


# Tratamento do dataframe final para pegar apenas as informações que queremos

In [56]:
# Lista com o nome exato das colunas de maior relevância
colunas_principais = [
    'srp', 'ano', 'numeroCompra', 'modalidadeNome',
    'unidadeOrgaoUfSigla',
    'unidadeOrgaoMunicipioNome',  
    'unidadeOrgaoCodigoIbge',     
    'orgaoEntidadeRazaoSocial', 'objetoCompra',
    'valorTotalEstimado', 'valorTotalHomologado', 'dataPublicacaoPncp',
]

# Seleciona apenas as colunas que realmente existem no df_total
colunas_presentes = [c for c in colunas_principais if c in df_total.columns]

# Cria um novo df para não mexer no original
df_final_reduzido = df_total[colunas_presentes].copy()

### Tratamento dos dados

In [57]:
# Converte colunas financeiras para numérico (Float)
for col_valor in ['valorTotalEstimado', 'valorTotalHomologado']:
    if col_valor in df_final_reduzido.columns:
        df_final_reduzido[col_valor] = pd.to_numeric(
            df_final_reduzido[col_valor], errors='coerce'
        )

# Converte a data de publicação para formato de data real
if 'dataPublicacaoPncp' in df_final_reduzido.columns:
    df_final_reduzido['dataPublicacaoPncp'] = pd.to_datetime(
        df_final_reduzido['dataPublicacaoPncp'], errors='coerce'
    )

# Falta ver oq vai fazer com o valorTotalHomologado, alguns estão vindo como NaN


### Remoção da modalidade "Pregão - Presencial"
Como são poucos dados e diferem do que queremos, foi exlcuido a modalidade que é irrelevante

In [58]:
## Remoção da modalidade "Pregão - Presencial" 451
# Como são poucos dados e diferem do que queremos, foi excluída a modalidade que é irrelevante
df_final_reduzido = df_final_reduzido[df_final_reduzido['modalidadeNome'] != 'Pregão - Presencial'].copy()

print(f"Registros após remoção: {len(df_final_reduzido)}")
print(df_final_reduzido['modalidadeNome'].value_counts())

Registros após remoção: 268524
modalidadeNome
Pregão - Eletrônico    134585
Dispensa               133939
Name: count, dtype: int64


## Dataframe final tratado

In [59]:
print("=== DATAFRAME FINAL REDUZIDO ===")
print(f"Dimensões do DataFrame: {df_final_reduzido.shape}")
display(df_final_reduzido.head(10))

=== DATAFRAME FINAL REDUZIDO ===
Dimensões do DataFrame: (268524, 12)


,srp,ano,numeroCompra,modalidadeNome,unidadeOrgaoUfSigla,unidadeOrgaoMunicipioNome,unidadeOrgaoCodigoIbge,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalEstimado,valorTotalHomologado,dataPublicacaoPncp
0,True,2024,00017,Pregão - Eletrônico,SP,SÃO PAULO,3550308,MINISTERIO DA FAZENDA,"Contratação, através de Ata de Registro de Pre...",614963.83,343793.50,2024-01-02 07:00:02
1,False,2024,00026,Pregão - Eletrônico,DF,BRASÍLIA,5300108,AGENCIA NACIONAL DE TELECOMUNICACOES,Contratação de empresa visando a cessão de uso...,404.74,NaN,2024-01-02 07:00:04
2,False,2024,00049,Pregão - Eletrônico,RS,PELOTAS,4314407,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",Prestação de serviços de manutenção preventiva...,105000.00,105000.00,2024-01-02 07:00:05
3,False,2024,00036,Pregão - Eletrônico,SP,PIRASSUNUNGA,3539301,COMANDO DA AERONAUTICA,O objeto da presente licitação é a Contratação...,286011.82,270000.00,2024-01-02 07:00:07
4,False,2024,00011,Pregão - Eletrônico,SP,SÃO PAULO,3550308,INSTITUTO NACIONAL DO SEGURO SOCIAL,Contratação dos serviços de administração e ge...,232490.96,NaN,2024-01-02 07:00:09
5,False,2024,00022,Pregão - Eletrônico,MG,BURITIS,3109303,TRIBUNAL DE CONTAS DO ESTADO DE RONDONIA,Contratação de serviço de administração e gere...,99975.00,95466.00,2024-01-02 07:00:10
6,False,2024,00008,Pregão - Eletrônico,RS,CAXIAS DO SUL,4305108,MUNICIPIO DE CAXIAS DO SUL,"Constitui o objeto do presente certame, a cont...",67770.00,65820.00,2024-01-02 07:00:12
7,True,2024,90001,Pregão - Eletrônico,RS,CAXIAS DO SUL,4305108,MUNICIPIO DE CAXIAS DO SUL,Fornecimento de materiais de Higiene e Limpeza...,68469.70,27300.60,2024-01-02 07:00:16
8,False,2024,23033,Pregão - Eletrônico,MG,BELO HORIZONTE,3106200,MUNICIPIO DE BELO HORIZONTE,"Aquisição de Caixa Acústica, Amplificada, Port...",35575.68,23196.00,2024-01-02 07:00:18
9,True,2024,00096,Pregão - Eletrônico,RS,PELOTAS,4314407,EMPRESA BRASILEIRA DE SERVIÇOS HOSPITALARES,Aquisição de Pulseira Hospitalar e Etiquetas.,137987.47,103025.62,2024-01-02 07:00:20


In [60]:
df_final_reduzido.to_parquet("srp_contratacoes_tratado_pais.parquet", index=False) # Salva a tabela final em parquet

In [61]:
df_conferencia = pd.read_parquet("srp_contratacoes_tratado_pais.parquet") # Lê a tabela final salva em parquet

## Tamanho final da Tabela

In [62]:
print(df_conferencia.shape)

(268524, 12)
